# Phase Diagram — Transition Sharpness Across Models

Explores whether there is a critical boundary in model-parameter space separating
**sharp phase transitions** (p109-like: fast second descent, near-simultaneous eff-dim
compression and grokking) from **supercritical/gradual transitions** (p59-like:
slow asymptotic descent, long gap between compression and grokking).

All data from `variant_summary.json` — no artifact loading required.

## Candidate axes
- `neurons_per_frequency` (npf) = d_mlp / n_committed_frequencies
- `frequency_spread` = max(committed_freq) - min(committed_freq)
- `prime` and `n_freqs` directly

## Sharpness metrics
- `descent_duration` = grokking_epoch - second_descent_onset_epoch (shorter = sharper)
- `two_step_lag` = grokking_epoch - eff_dim_crossover_epoch (smaller = more simultaneous)
- Variants that never crossed the grokking threshold within training are treated separately


In [1]:
import json
import pathlib

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RESULTS_DIR = pathlib.Path("../results/modulo_addition_1layer")
EXPORT_DIR = pathlib.Path("exports/phase_diagram")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
def load_variant_rows(results_dir):
    rows = []
    for summary_path in sorted(results_dir.glob("*/variant_summary.json")):
        d = json.load(open(summary_path))
        cfg_path = summary_path.parent / "config.json"
        cfg = json.load(open(cfg_path)) if cfg_path.exists() else {}

        fw = d.get("final_window", {})
        committed = fw.get("committed_frequencies_end") or []

        clf = d.get("performance_classification", ["unknown", []])
        label = clf[0] if isinstance(clf, list) else str(clf)

        grok = d.get("test_loss_threshold_first_epoch")
        grok = grok if (grok and grok > 0) else None
        onset = d.get("second_descent_onset_epoch")
        eff_xo = d.get("effective_dimensionality_cross_over_epoch")

        prime = d.get("prime")
        d_mlp = cfg.get("d_mlp", 512)
        n_freqs = len(committed)

        rows.append({
            "prime": prime,
            "model_seed": d.get("model_seed"),
            "data_seed": d.get("data_seed"),
            "variant": f"p{prime}/s{d.get('model_seed')}/ds{d.get('data_seed')}",
            "d_mlp": d_mlp,
            "label": label,
            "n_freqs": n_freqs,
            "committed_freqs": sorted(committed),
            "freq_spread": (max(committed) - min(committed)) if n_freqs > 1 else 0,
            "npf": d_mlp / n_freqs if n_freqs > 0 else None,
            "grokking_epoch": grok,
            "second_descent_onset": onset,
            "eff_dim_crossover": eff_xo,
            "first_mover_epoch": d.get("first_mover_epoch"),
            "descent_duration": (grok - onset) if (grok and onset) else None,
            "two_step_lag": (grok - eff_xo) if (grok and eff_xo) else None,
            "test_loss_final": d.get("test_loss_final"),
            "max_circularity": d.get("max_resid_post_circularity"),
            "grokked": grok is not None,
        })

    return pd.DataFrame(rows)


df = load_variant_rows(RESULTS_DIR)
print(f"{len(df)} variants loaded")
print(f"  grokked within window: {df['grokked'].sum()} / {len(df)}")
print(f"  labels: {df['label'].value_counts().to_dict()}")

30 variants loaded
  grokked within window: 20 / 30
  labels: {'healthy': 14, 'late_grokker': 10, 'degraded': 5, 'no_second_descent': 1}


In [3]:
display_cols = [
    "variant", "label", "n_freqs", "npf", "freq_spread",
    "second_descent_onset", "grokking_epoch", "descent_duration",
    "two_step_lag", "max_circularity",
]
df_display = df[display_cols].copy()
df_display["npf"] = df_display["npf"].apply(lambda x: f"{x:.0f}" if x else "—")
df_display["descent_duration"] = df_display["descent_duration"].apply(lambda x: f"{x:.0f}" if x else "—")
df_display["two_step_lag"] = df_display["two_step_lag"].apply(lambda x: f"{x:.0f}" if x else "—")
df_display["max_circularity"] = df_display["max_circularity"].apply(lambda x: f"{x:.3f}")
df_display.sort_values("variant")

,variant,label,n_freqs,npf,freq_spread,second_descent_onset,grokking_epoch,descent_duration,two_step_lag,max_circularity
0,p101/s485/ds42,late_grokker,0,nan,0,16820.0,NaN,nan,nan,0.839
1,p101/s485/ds598,late_grokker,4,128,34,14045.0,16656.0,2611,656,0.428
2,p101/s485/ds999,degraded,3,171,31,6463.0,NaN,nan,nan,0.566
3,p101/s999/ds42,healthy,4,128,37,10146.0,13201.0,3055,801,0.642
4,p101/s999/ds598,late_grokker,0,nan,0,13043.0,NaN,nan,nan,0.367
5,p101/s999/ds999,healthy,3,171,24,5037.0,10036.0,4999,3036,0.619
6,p103/s485/ds598,healthy,4,128,34,7699.0,9489.0,1790,-11,0.485
7,p103/s999/ds598,healthy,4,128,24,4474.0,6716.0,2242,216,0.679
8,p107/s485/ds598,late_grokker,4,128,31,16781.0,20308.0,3527,1008,0.559
9,p107/s999/ds598,healthy,3,171,29,9217.0,12249.0,3032,449,0.929


## Main phase diagram — neurons-per-frequency vs frequency spread

These two axes capture different aspects of how the model distributes resources:
- **npf** (neurons per frequency): how many neurons are available per Fourier mode
- **freq_spread**: how wide the committed frequency set is across the spectrum

Color encodes `descent_duration` (grokking epoch - second descent onset).
Models that never crossed the grokking threshold are shown as open markers (×).

In [4]:
_LABEL_COLORS = {
    "healthy": "#2196F3",
    "late_grokker": "#FF9800",
    "degraded": "#9C27B0",
    "no_second_descent": "#F44336",
    "unknown": "#9E9E9E",
}

df_grokked = df[df["grokked"] & df["npf"].notna()].copy()
df_ungrokked = df[~df["grokked"] & df["npf"].notna()].copy()

fig = go.Figure()

# Grokked variants — color by descent_duration
fig.add_trace(go.Scatter(
    x=df_grokked["npf"],
    y=df_grokked["freq_spread"],
    mode="markers",
    marker=dict(
        size=12,
        color=df_grokked["descent_duration"],
        colorscale="RdYlGn_r",  # red = long duration (gradual), green = short (sharp)
        colorbar=dict(title="Descent<br>duration", thickness=14, len=0.7),
        line=dict(width=1, color="white"),
        cmin=1000,
        cmax=7000,
    ),
    text=df_grokked["variant"] + "<br>" + df_grokked["label"] +
         "<br>dur=" + df_grokked["descent_duration"].apply(lambda x: f"{x:.0f}" if x else "—"),
    hovertemplate="%{text}<br>npf=%{x:.0f} spread=%{y}<extra></extra>",
    name="grokked",
    showlegend=True,
))

# Ungrokked variants — grey × markers
if len(df_ungrokked) > 0:
    fig.add_trace(go.Scatter(
        x=df_ungrokked["npf"],
        y=df_ungrokked["freq_spread"],
        mode="markers",
        marker=dict(size=12, symbol="x", color="#BDBDBD", line=dict(width=2)),
        text=df_ungrokked["variant"] + "<br>" + df_ungrokked["label"],
        hovertemplate="%{text}<br>npf=%{x:.0f} spread=%{y}<extra>no grokking threshold</extra>",
        name="no grokking threshold",
    ))

# Annotate reference models
references = [
    ("p109/s485/ds598", "p109\n(sharp)"),
    ("p59/s485/ds598", "p59\n(supercritical)"),
    ("p113/s999/ds598", "p113\n(canon)"),
    ("p101/s999/ds598", "p101\n(open-loop)"),
]
for variant_id, annotation in references:
    row = df[df["variant"] == variant_id]
    if len(row) == 0 or row["npf"].isna().all():
        continue
    r = row.iloc[0]
    fig.add_annotation(
        x=r["npf"], y=r["freq_spread"],
        text=annotation.replace("\n", "<br>"),
        showarrow=True, arrowhead=2, arrowsize=0.8,
        ax=30, ay=-30, font=dict(size=10),
    )

fig.update_layout(
    title="Phase diagram — neurons per frequency vs frequency spread<br>"
          "<sup>Color = descent duration (red=gradual, green=sharp) | × = no grokking threshold crossed</sup>",
    xaxis_title="Neurons per frequency (d_mlp / n_committed_freqs)",
    yaxis_title="Frequency spread (max − min committed freq)",
    template="plotly_white",
    height=550,
    width=800,
)
fig.show()
fig.write_image(str(EXPORT_DIR / "phase_diagram_npf_spread.png"))

## Phase diagram — prime vs n_committed_frequencies

Direct model parameter space. Color = descent_duration as above.
Jitter applied to separate overlapping points (same prime, different seeds/data_seeds).

In [5]:
rng = np.random.default_rng(42)

fig = go.Figure()

for grokked, marker_symbol, name in [(True, "circle", "grokked"), (False, "x", "no grokking threshold")]:
    sub = df[df["grokked"] == grokked].copy()
    if len(sub) == 0:
        continue

    jitter_x = rng.uniform(-0.4, 0.4, len(sub))
    jitter_y = rng.uniform(-0.1, 0.1, len(sub))

    color = sub["descent_duration"] if grokked else ["#BDBDBD"] * len(sub)
    colorscale = "RdYlGn_r" if grokked else None
    showscale = grokked

    fig.add_trace(go.Scatter(
        x=(sub["prime"] + jitter_x).tolist(),
        y=(sub["n_freqs"] + jitter_y).tolist(),
        mode="markers",
        marker=dict(
            size=12,
            symbol=marker_symbol,
            color=color,
            colorscale=colorscale,
            showscale=showscale,
            colorbar=dict(title="Descent<br>duration", thickness=14, len=0.7) if showscale else None,
            cmin=1000, cmax=7000,
            line=dict(width=1 if grokked else 2, color="white" if grokked else None),
        ),
        text=sub["variant"] + "<br>" + sub["label"] +
             "<br>dur=" + sub["descent_duration"].apply(lambda x: f"{x:.0f}" if x else "—"),
        hovertemplate="%{text}<br>p=%{x:.0f} n_freqs=%{y:.0f}<extra></extra>",
        name=name,
        showlegend=True,
    ))

fig.update_layout(
    title="Phase diagram — prime vs n_committed_frequencies<br>"
          "<sup>Color = descent duration | × = no grokking threshold | jitter applied for readability</sup>",
    xaxis=dict(title="prime (p)", tickvals=sorted(df["prime"].unique())),
    yaxis=dict(title="n committed frequencies", tickvals=list(range(6))),
    template="plotly_white",
    height=500,
    width=800,
)
fig.show()
fig.write_image(str(EXPORT_DIR / "phase_diagram_prime_nfreqs.png"))

## Sharpness distributions by label

Box plots of `descent_duration` and `two_step_lag` grouped by performance label.
Models that never grokked are excluded (no duration defined).

In [6]:
df_with_dur = df[df["descent_duration"].notna()].copy()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Descent duration", "Two-step lag (grokking − eff_dim crossover)"],
    horizontal_spacing=0.12,
)

label_order = ["healthy", "late_grokker", "degraded", "no_second_descent"]
present_labels = [l for l in label_order if l in df_with_dur["label"].values]

for col_idx, metric in enumerate(["descent_duration", "two_step_lag"], start=1):
    sub = df_with_dur[df_with_dur[metric].notna()]
    for lbl in present_labels:
        vals = sub[sub["label"] == lbl][metric].tolist()
        if not vals:
            continue
        color = _LABEL_COLORS.get(lbl, "#9E9E9E")
        fig.add_trace(
            go.Box(
                y=vals,
                name=lbl,
                marker_color=color,
                boxpoints="all",
                jitter=0.3,
                pointpos=0,
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig.update_yaxes(title_text="Epochs", row=1, col=1)
fig.update_yaxes(title_text="Epochs", row=1, col=2)
fig.update_layout(
    title="Transition sharpness metrics by performance label",
    template="plotly_white",
    height=450,
    width=800,
    legend=dict(orientation="h", y=-0.15),
)
fig.show()
fig.write_image(str(EXPORT_DIR / "sharpness_by_label.png"))

## Two-step lag vs descent duration

Scatter of the two sharpness metrics against each other. If they're measuring the
same thing, points should fall along a diagonal. Divergence identifies models where
the eff-dim compression / grokking gap and the second-descent width are decoupled.

In [7]:
df_both = df[df["descent_duration"].notna() & df["two_step_lag"].notna()].copy()

fig = px.scatter(
    df_both,
    x="descent_duration",
    y="two_step_lag",
    color="label",
    color_discrete_map=_LABEL_COLORS,
    hover_data=["variant", "prime", "n_freqs", "npf", "freq_spread"],
    text="variant",
    title="Two-step lag vs descent duration<br>"
          "<sup>Diagonal = metrics track together; outliers = decoupled dynamics</sup>",
    labels={
        "descent_duration": "Descent duration (grokking − onset)",
        "two_step_lag": "Two-step lag (grokking − eff_dim crossover)",
    },
    template="plotly_white",
    height=500,
    width=700,
)
fig.update_traces(textposition="top center", textfont_size=8)

# Diagonal reference
lim = max(df_both["descent_duration"].max(), df_both["two_step_lag"].max()) * 1.05
fig.add_trace(go.Scatter(
    x=[0, lim], y=[0, lim],
    mode="lines",
    line=dict(color="rgba(0,0,0,0.15)", dash="dot"),
    showlegend=False,
))

fig.show()
fig.write_image(str(EXPORT_DIR / "two_step_vs_duration.png"))

## Circularity vs sharpness

Max residual-stream circularity is a different lens on how well the model
organizes its representation. Does higher circularity correlate with sharper transitions?

In [8]:
df_circ = df[df["descent_duration"].notna()].copy()

fig = px.scatter(
    df_circ,
    x="max_circularity",
    y="descent_duration",
    color="label",
    color_discrete_map=_LABEL_COLORS,
    hover_data=["variant", "prime", "n_freqs", "npf", "freq_spread"],
    text="variant",
    title="Max circularity vs descent duration<br>"
          "<sup>Higher circularity = more torus-like representation</sup>",
    labels={
        "max_circularity": "Max residual-stream circularity",
        "descent_duration": "Descent duration (shorter = sharper)",
    },
    template="plotly_white",
    height=450,
    width=650,
)
fig.update_traces(textposition="top center", textfont_size=8)
fig.show()
fig.write_image(str(EXPORT_DIR / "circularity_vs_duration.png"))

## Second descent onset timing

When does the second descent start, relative to the first mover epoch?
The gap between first-mover emergence and second-descent onset may indicate
how long the model needs to organize after the first frequency commits.
Models near the critical point should have a shorter gap.

In [9]:
df_onset = df[df["first_mover_epoch"].notna() & df["second_descent_onset"].notna()].copy()
df_onset["fm_to_onset_gap"] = df_onset["second_descent_onset"] - df_onset["first_mover_epoch"]

fig = px.scatter(
    df_onset,
    x="fm_to_onset_gap",
    y="descent_duration",
    color="label",
    color_discrete_map=_LABEL_COLORS,
    hover_data=["variant", "prime", "n_freqs", "second_descent_onset", "first_mover_epoch"],
    text="variant",
    title="First-mover → onset gap vs descent duration<br>"
          "<sup>x = time from first-frequency commitment to start of second descent</sup>",
    labels={
        "fm_to_onset_gap": "First-mover → second-descent onset gap (epochs)",
        "descent_duration": "Descent duration (shorter = sharper)",
    },
    template="plotly_white",
    height=450,
    width=700,
)
fig.update_traces(textposition="top center", textfont_size=8)
fig.show()
fig.write_image(str(EXPORT_DIR / "fm_gap_vs_duration.png"))

## Summary statistics

Quick numerical summary of the key metrics.

In [10]:
summary = df.groupby("label").agg(
    count=("variant", "count"),
    grokked_pct=("grokked", lambda x: f"{100*x.mean():.0f}%"),
    npf_median=("npf", lambda x: f"{x.median():.0f}" if x.notna().any() else "—"),
    spread_median=("freq_spread", "median"),
    dur_median=("descent_duration", lambda x: f"{x.median():.0f}" if x.notna().any() else "—"),
    lag_median=("two_step_lag", lambda x: f"{x.median():.0f}" if x.notna().any() else "—"),
    circ_mean=("max_circularity", lambda x: f"{x.mean():.3f}"),
).reset_index()

summary.columns = [
    "Label", "Count", "Grokked%", "NPF median", "Freq spread median",
    "Descent dur median", "Two-step lag median", "Circularity mean",
]
display(summary)

,Label,Count,Grokked%,NPF median,Freq spread median,Descent dur median,Two-step lag median,Circularity mean
0,degraded,5,0%,171,20.0,—,—,0.571
1,healthy,14,100%,171,25.0,2861,356,0.678
2,late_grokker,10,60%,128,23.5,3685,1182,0.571
3,no_second_descent,1,0%,—,0.0,—,—,0.600


## Early prediction: group geometry at the eff_dim crossover epoch

**Hypothesis (REQ_081 Notes):** The geometry at the eff_dim crossover epoch may already
contain signals about whether each neuron group will cleanly crystallize — before the
second descent completes.

Three per-group signals at crossover:

| signal | interpretation |
|---|---|
| PC1 >> PC2 (PC1 > 60%) | 1D degenerate collapse already underway — group will not form a ring |
| PC3 < 10%, PC1 ≈ PC2 | Ring already nearly closed — group is ahead of schedule, likely to succeed |
| High R (> 0.3) with PC3 > 20% | Angularly concentrated but not in a ring plane — contested/chaotic state |
| Low R (< 0.15), PC3 10–20% | Healthy in-progress — compression happened, crystallization not yet started |

In [ ]:
def _rayleigh_r(angles):
    return abs(np.mean(np.exp(1j * np.asarray(angles))))


def _crossover_group_signals(variant_dir: pathlib.Path) -> list[dict] | None:
    """Per-group geometry signals at the eff_dim crossover epoch."""
    summary_path = variant_dir / "variant_summary.json"
    if not summary_path.exists():
        return None
    summary = json.load(open(summary_path))
    xo_epoch = summary.get("effective_dimensionality_cross_over_epoch")
    if not xo_epoch:
        return None

    ngpca_path = variant_dir / "artifacts" / "neuron_group_pca" / "cross_epoch.npz"
    if not ngpca_path.exists():
        return None

    ngpca = np.load(str(ngpca_path), allow_pickle=True)
    epochs = ngpca["epochs"]
    projections = ngpca["projections"]   # (n_epochs, d_mlp, 3)
    neuron_group_idx = ngpca["neuron_group_idx"]
    group_freqs = ngpca["group_freqs"]
    pc_var = ngpca["pc_var"]             # (n_epochs, n_groups, 3)

    ep_idx = int(np.argmin(np.abs(epochs.astype(int) - int(xo_epoch))))
    projs = projections[ep_idx]

    rows = []
    for g_idx in range(len(group_freqs)):
        members = np.where(neuron_group_idx == g_idx)[0]
        coords = projs[members]
        valid = ~np.isnan(coords[:, 0])
        if valid.sum() < 5:
            continue
        angles = np.arctan2(coords[valid, 1], coords[valid, 0])
        r = _rayleigh_r(angles)
        pv = pc_var[ep_idx, g_idx]
        total = pv.sum()
        pc1, pc2, pc3 = pv[0] / total, pv[1] / total, pv[2] / total

        # Classify signal
        if pc1 > 0.60:
            signal = "degenerate_1d"
        elif pc3 < 0.10 and abs(pc1 - pc2) < 0.10:
            signal = "ring_closing"
        elif r > 0.30 and pc3 > 0.20:
            signal = "contested_chaotic"
        else:
            signal = "healthy_in_progress"

        rows.append({
            "freq": int(group_freqs[g_idx]) + 1,
            "n": int(valid.sum()),
            "R": float(r),
            "pc1": float(pc1),
            "pc2": float(pc2),
            "pc3": float(pc3),
            "signal": signal,
            "xo_epoch": int(epochs[ep_idx]),
        })
    return rows


# Run on reference models
REFERENCE_MODELS = [
    ("modulo_addition_1layer_p109_seed485_dseed598", "p109/485/598", "healthy"),
    ("modulo_addition_1layer_p101_seed485_dseed999", "p101/485/999", "stop"),
    ("modulo_addition_1layer_p113_seed485_dseed999", "p113/485/999", "stop"),
    ("modulo_addition_1layer_p113_seed999_dseed999", "p113/999/999", "stop"),
    ("modulo_addition_1layer_p59_seed485_dseed598",  "p59/485/598",  "continue"),
    ("modulo_addition_1layer_p101_seed485_dseed42",  "p101/485/42",  "healthy"),
]

_SIGNAL_COLORS = {
    "ring_closing":       "#2196F3",
    "healthy_in_progress":"#4CAF50",
    "degenerate_1d":      "#F44336",
    "contested_chaotic":  "#FF9800",
}

fig = make_subplots(
    rows=len(REFERENCE_MODELS), cols=1,
    subplot_titles=[f"{label} ({outcome})" for _, label, outcome in REFERENCE_MODELS],
    vertical_spacing=0.06,
)

for row_idx, (suffix, label, outcome) in enumerate(REFERENCE_MODELS, start=1):
    vdir = RESULTS_DIR / suffix
    rows = _crossover_group_signals(vdir)
    if not rows:
        continue
    for r in rows:
        color = _SIGNAL_COLORS.get(r["signal"], "#9E9E9E")
        fig.add_trace(go.Bar(
            x=[f"freq {r['freq']}\n(n={r['n']})"],
            y=[r["pc3"]],
            name=r["signal"],
            marker_color=color,
            text=[f"R={r['R']:.2f}<br>PC1={r['pc1']:.0%}"],
            textposition="outside",
            showlegend=(row_idx == 1),
            legendgroup=r["signal"],
        ), row=row_idx, col=1)

    fig.add_hline(y=0.10, line_dash="dot", line_color="rgba(0,0,0,0.2)",
                  annotation_text="PC3=10% threshold", row=row_idx, col=1)
    fig.update_yaxes(title_text="PC3 fraction", range=[0, 0.45], row=row_idx, col=1)

fig.update_layout(
    title=(
        "Per-group geometry at eff_dim crossover epoch<br>"
        "<sup>PC3 height = how open the ring is | color = signal type | "
        "blue=ring closing, green=healthy in-progress, red=degenerate 1D, orange=contested</sup>"
    ),
    template="plotly_white",
    height=220 * len(REFERENCE_MODELS),
    width=700,
    bargap=0.3,
    legend=dict(orientation="h", y=-0.04),
)
fig.show()
fig.write_image(str(EXPORT_DIR / "crossover_group_signals.png"))

## Effective dimensionality — two-phase slope analysis

The activation-space effective dimensionality (`resid_post_mean_dim` from `repr_geometry`)
shows two distinct steep descent phases for every model:

1. **First descent** — early memorization; the model compresses rapidly from the start
2. **Second descent** — structural transition (the grokking compression event)

Between them: a **plateau** where dimensionality is roughly stable.

Metrics extracted per variant:
- `second_descent_slope` — steepness of the second compression (more negative = sharper transition)
- `plateau_width` — epochs between first and second descent peaks
- `eff_dim_final` — residual dimensionality at end of training

Note: weight-space W_in participation ratio was also tested but shows near-identical
second descent slopes (~-0.014) across all variants — it does not discriminate sharp from
supercritical transitions. The activation-space signal is the right lens here.

In [ ]:
from miscope.analysis.artifact_loader import ArtifactLoader


def _compute_two_phase_slopes(variant_dir: pathlib.Path) -> dict | None:
    loader = ArtifactLoader(str(variant_dir / "artifacts"))
    try:
        epochs = sorted(loader.get_epochs("repr_geometry"))
    except Exception:
        return None
    if len(epochs) < 10:
        return None

    stacked = loader.load_epochs("repr_geometry", fields=["resid_post_mean_dim"])
    eff_dim = stacked["resid_post_mean_dim"]
    epochs_arr = np.array(epochs, dtype=float)

    dt = np.diff(epochs_arr)
    d_eff = np.diff(eff_dim) / dt
    epoch_mid = (epochs_arr[:-1] + epochs_arr[1:]) / 2

    # Two steepest descent points, separated by at least 2000 epochs
    i_sorted = np.argsort(d_eff)
    i1 = i_sorted[0]
    i2 = None
    for i in i_sorted[1:]:
        if abs(epoch_mid[i] - epoch_mid[i1]) > 2000:
            i2 = i
            break

    # Ensure chronological order
    if i2 is not None and epoch_mid[i1] > epoch_mid[i2]:
        i1, i2 = i2, i1

    return {
        "eff_dim_initial": float(eff_dim[0]),
        "eff_dim_final": float(eff_dim[-1]),
        "first_descent_slope": float(d_eff[i1]),
        "first_descent_epoch": float(epoch_mid[i1]),
        "second_descent_slope": float(d_eff[i2]) if i2 is not None else None,
        "second_descent_epoch": float(epoch_mid[i2]) if i2 is not None else None,
        "plateau_width": float(epoch_mid[i2] - epoch_mid[i1]) if i2 is not None else None,
    }


# Compute for all variants and merge into df
slope_rows = []
for summary_path in sorted(RESULTS_DIR.glob("*/variant_summary.json")):
    d = json.load(open(summary_path))
    variant_id = f"p{d['prime']}/s{d['model_seed']}/ds{d['data_seed']}"
    metrics = _compute_two_phase_slopes(summary_path.parent)
    if metrics:
        slope_rows.append({"variant": variant_id, **metrics})

df_slopes = pd.DataFrame(slope_rows)
df = df.merge(df_slopes, on="variant", how="left")

# bounce_magnitude from variant_summary
bounce_rows = []
for summary_path in sorted(RESULTS_DIR.glob("*/variant_summary.json")):
    d = json.load(open(summary_path))
    variant_id = f"p{d['prime']}/s{d['model_seed']}/ds{d['data_seed']}"
    loss_min = d.get("test_loss_min")
    loss_final = d.get("test_loss_final")
    bounce = (loss_final - loss_min) if (loss_min is not None and loss_final is not None) else None
    bounce_rows.append({"variant": variant_id, "bounce_magnitude": bounce})

df = df.merge(pd.DataFrame(bounce_rows), on="variant", how="left")

print(f"DataFrame now has {len(df.columns)} columns")
print(df[["variant", "label", "second_descent_slope", "plateau_width", "bounce_magnitude"]].to_string(index=False))

## Enhanced phase diagram — second descent slope vs plateau width

Now every variant (including never-grokked) has a sharpness signal.
X-axis: `second_descent_slope` (more negative = sharper structural transition)
Y-axis: `plateau_width` (wider = longer wait between memorization compression and grokking compression)
Color: `bounce_magnitude` (test_loss_final − test_loss_min — how much the model deteriorated after its best point)

In [ ]:
df_slope_valid = df[df["second_descent_slope"].notna() & df["plateau_width"].notna()].copy()

fig = go.Figure()

for lbl in ["healthy", "late_grokker", "degraded", "no_second_descent", "unknown"]:
    sub = df_slope_valid[df_slope_valid["label"] == lbl]
    if len(sub) == 0:
        continue
    color = _LABEL_COLORS.get(lbl, "#9E9E9E")
    fig.add_trace(go.Scatter(
        x=sub["second_descent_slope"].tolist(),
        y=sub["plateau_width"].tolist(),
        mode="markers+text",
        name=lbl,
        marker=dict(
            size=14,
            color=sub["bounce_magnitude"].tolist(),
            colorscale="Reds",
            cmin=0,
            cmax=df_slope_valid["bounce_magnitude"].quantile(0.9),
            colorbar=dict(title="Bounce<br>magnitude", thickness=14, len=0.6),
            line=dict(width=1.5, color=color),
            symbol="circle",
        ),
        text=sub["variant"].str.replace("/s", "<br>s").str.replace("/ds", "/ds"),
        textposition="top center",
        textfont=dict(size=8),
        hovertemplate=(
            "%{text}<br>"
            "2nd slope=%{x:.5f}<br>"
            "plateau=%{y:.0f}<br>"
            "bounce=%{marker.color:.4f}"
            "<extra>" + lbl + "</extra>"
        ),
    ))

# Annotate reference models
for variant_id, label_txt in [
    ("p109/s485/ds598", "p109 (sharp)"),
    ("p59/s485/ds598", "p59 (supercritical)"),
    ("p113/s999/ds598", "p113 (canon)"),
    ("p101/s999/ds598", "p101 (open-loop)"),
]:
    row = df_slope_valid[df_slope_valid["variant"] == variant_id]
    if len(row) == 0:
        continue
    r = row.iloc[0]
    fig.add_annotation(
        x=r["second_descent_slope"], y=r["plateau_width"],
        text=label_txt, showarrow=True, arrowhead=2,
        ax=40, ay=-35, font=dict(size=10, color="#333"),
    )

fig.update_layout(
    title=(
        "Enhanced phase diagram — eff_dim second descent slope vs plateau width<br>"
        "<sup>Marker outline color = performance label | Fill color = bounce magnitude (red = large rebound)</sup>"
    ),
    xaxis=dict(title="Second descent slope (more negative = sharper)", autorange="reversed"),
    yaxis_title="Plateau width (epochs between first and second steep descent)",
    template="plotly_white",
    height=600,
    width=900,
    legend=dict(orientation="h", y=-0.12),
)
fig.show()
fig.write_image(str(EXPORT_DIR / "phase_diagram_slope_plateau.png"))